<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/02_react_code_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
!pip install -q -U google-genai

In [20]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.6-flash'

def ask(user_prompt, system=None, temperature=0.2, model=MODEL):
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    resp = client.models.generate_content(
        model=model, contents=user_prompt,
        config=types.GenerateContentConfig(**kwargs),
    )
    return (resp.text or '').strip()

print('Connected. Using:', MODEL)

Connected. Using: gemini-3.6-flash


In [21]:
REACT_SYSTEM = """You are a Python engineer working inside an explicit ReAct loop.

Every turn you output exactly these four sections, in this order, with these exact headers:

THOUGHT:
  What you know, what the last OBSERVATION changed about that, and what you will do differently
  this turn. 3 to 6 lines. No code in this section.

PLAN:
  Numbered implementation steps, maximum 5.

ACTION:
  Exactly one ```python fenced block containing the complete runnable module. Never a diff, never
  a fragment, never two blocks. Rewrite the whole module every turn.

EXPECTATION:
  One line per behavior you are genuinely unsure the harness will accept.

Hard rules:
- The OBSERVATION you receive is the real stdout of a real test harness that already executed your
  code. It is ground truth. Never argue with it and never explain it away.
- Never state that a test passes unless the OBSERVATION told you it passed.
- Python standard library only. No pandas, no numpy, no third-party packages.
- No prose outside the four sections."""

print(REACT_SYSTEM)

You are a Python engineer working inside an explicit ReAct loop.

Every turn you output exactly these four sections, in this order, with these exact headers:

THOUGHT:
  What you know, what the last OBSERVATION changed about that, and what you will do differently
  this turn. 3 to 6 lines. No code in this section.

PLAN:
  Numbered implementation steps, maximum 5.

ACTION:
  Exactly one ```python fenced block containing the complete runnable module. Never a diff, never
  a fragment, never two blocks. Rewrite the whole module every turn.

EXPECTATION:
  One line per behavior you are genuinely unsure the harness will accept.

Hard rules:
- The OBSERVATION you receive is the real stdout of a real test harness that already executed your
  code. It is ground truth. Never argue with it and never explain it away.
- Never state that a test passes unless the OBSERVATION told you it passed.
- Python standard library only. No pandas, no numpy, no third-party packages.
- No prose outside the four 

In [22]:
TASK = """Write a Python module for the VNTG OS consignor payout calculator.

FUNCTION
  calculate_payouts(sales: list[dict]) -> dict

INPUT
  Each element of `sales` is a dict with exactly these keys:
    item_id          str     e.g. "VN-0184"
    consignor_id     str     e.g. "C-207"
    sale_price       str     dollars as a string, e.g. "129.99"
    status           str     one of "sold", "returned", "pending"
    days_since_sale  int     whole days since the sale closed

BUSINESS RULES
  1. A sale is payable only if status == "sold" AND days_since_sale >= 7
     (7 days is the buyer return window).
  2. The consignor's share is 60% of sale_price, or 65% if sale_price is 200 dollars or more.
  3. All money is handled with decimal.Decimal and must be exact to the cent.

OUTPUT
  A dict keyed by consignor_id. Each value is a dict with exactly:
    "item_count"   int
    "gross_sales"  Decimal, quantized to 2 places, sum of payable sale prices
    "payout"       Decimal, quantized to 2 places, what the consignor is owed
  Consignors with no payable items must not appear in the output.

LIBRARIES
  Python standard library only. decimal and typing are available. No pandas, no numpy.

FORMATTING
  One module-level function calculate_payouts plus any private helpers. Type hints on every
  signature. A short docstring on the public function. Code only, inside one ```python block.

ERROR HANDLING
  Invalid input must fail loudly and specifically rather than silently producing a wrong number.
"""

print(TASK)

Write a Python module for the VNTG OS consignor payout calculator.

FUNCTION
  calculate_payouts(sales: list[dict]) -> dict

INPUT
  Each element of `sales` is a dict with exactly these keys:
    item_id          str     e.g. "VN-0184"
    consignor_id     str     e.g. "C-207"
    sale_price       str     dollars as a string, e.g. "129.99"
    status           str     one of "sold", "returned", "pending"
    days_since_sale  int     whole days since the sale closed

BUSINESS RULES
  1. A sale is payable only if status == "sold" AND days_since_sale >= 7
     (7 days is the buyer return window).
  2. The consignor's share is 60% of sale_price, or 65% if sale_price is 200 dollars or more.
  3. All money is handled with decimal.Decimal and must be exact to the cent.

OUTPUT
  A dict keyed by consignor_id. Each value is a dict with exactly:
    "item_count"   int
    "gross_sales"  Decimal, quantized to 2 places, sum of payable sale prices
    "payout"       Decimal, quantized to 2 places, 

In [23]:
from decimal import Decimal
import traceback, re

def _sale(item, consignor, price, status='sold', days=30):
    return {'item_id': item, 'consignor_id': consignor, 'sale_price': price,
            'status': status, 'days_since_sale': days}

def t_empty_input(f):
    assert f([]) == {}, f'expected {{}} for no sales, got {f([])!r}'

def t_basic_split(f):
    out = f([_sale('VN-1', 'C-1', '100.00')])
    assert out['C-1']['payout'] == Decimal('60.00'), out['C-1']['payout']
    assert out['C-1']['gross_sales'] == Decimal('100.00'), out['C-1']['gross_sales']
    assert out['C-1']['item_count'] == 1, out['C-1']['item_count']

def t_return_window(f):
    day3 = f([_sale('VN-2', 'C-1', '80.00', days=3)])
    assert day3 == {}, f'day 3 is inside the return window, should not pay: {day3!r}'
    day7 = f([_sale('VN-3', 'C-1', '80.00', days=7)])
    assert day7['C-1']['payout'] == Decimal('48.00'), day7['C-1']['payout']

def t_returned_excluded(f):
    out = f([_sale('VN-4', 'C-1', '90.00', status='returned')])
    assert out == {}, f'a returned item must never pay out: {out!r}'

def t_tier_boundary(f):
    out = f([_sale('VN-5', 'C-2', '200.00')])
    assert out['C-2']['payout'] == Decimal('130.00'), \
        f"$200.00 is the 65% tier (inclusive): expected 130.00, got {out['C-2']['payout']}"

def t_rounding_half_up(f):
    out = f([_sale('VN-6', 'C-2', '200.10')])
    assert out['C-2']['payout'] == Decimal('130.07'), \
        f"200.10 * 0.65 = 130.065 exactly, round half up to 130.07, got {out['C-2']['payout']}"

def t_negative_price(f):
    try:
        f([_sale('VN-7', 'C-3', '-20.00')])
    except ValueError:
        return
    except Exception as e:
        raise AssertionError(f'expected ValueError on a negative price, got {type(e).__name__}')
    raise AssertionError('a negative sale price was accepted silently')

def t_two_consignors(f):
    out = f([_sale('VN-8', 'C-1', '50.00'), _sale('VN-9', 'C-1', '25.00'),
             _sale('VN-10', 'C-4', '300.00'), _sale('VN-11', 'C-4', '10.00', days=1)])
    assert out['C-1']['item_count'] == 2, out['C-1']['item_count']
    assert out['C-1']['payout'] == Decimal('45.00'), out['C-1']['payout']
    assert out['C-4']['item_count'] == 1, 'the 1-day-old sale is not payable yet'
    assert out['C-4']['payout'] == Decimal('195.00'), out['C-4']['payout']

def t_returns_decimal(f):
    out = f([_sale('VN-12', 'C-1', '100.00')])
    assert isinstance(out['C-1']['payout'], Decimal), \
        f"payout must be Decimal, got {type(out['C-1']['payout']).__name__}"

TESTS = [
    ('empty_input', t_empty_input),
    ('basic_split', t_basic_split),
    ('return_window', t_return_window),
    ('returned_excluded', t_returned_excluded),
    ('tier_boundary', t_tier_boundary),
    ('rounding_half_up', t_rounding_half_up),
    ('negative_price', t_negative_price),
    ('two_consignors', t_two_consignors),
    ('returns_decimal', t_returns_decimal),
]
print(len(TESTS), 'acceptance tests loaded')

9 acceptance tests loaded


### The harness

In [24]:
def extract_code(reply: str) -> str:
    """Pull the single python block out of an ACTION section."""
    blocks = re.findall(r'```(?:python)?\n(.*?)```', reply, re.DOTALL)
    if not blocks:
        raise ValueError('no fenced python block found in the reply')
    return blocks[0].strip()

def run_acceptance(code_str: str):
    """Execute the generated module and run every test. Returns (all_passed, report)."""
    ns = {}
    try:
        exec(code_str, ns)
    except Exception:
        return False, 'MODULE FAILED TO EXECUTE:\n' + traceback.format_exc(limit=2)
    if 'calculate_payouts' not in ns:
        return False, 'No function named calculate_payouts was defined.'
    fn = ns['calculate_payouts']
    lines, passed = [], True
    for name, test in TESTS:
        try:
            test(fn)
            lines.append(f'PASS   {name}')
        except AssertionError as e:
            passed = False
            lines.append(f'FAIL   {name}: {e}')
        except Exception as e:
            passed = False
            lines.append(f'ERROR  {name}: {type(e).__name__}: {e}')
    score = sum(l.startswith('PASS') for l in lines)
    lines.append(f'--- {score}/{len(TESTS)} passing ---')
    return passed, '\n'.join(lines)

print('harness ready')

harness ready


In [25]:
transcript = []
conversation = TASK
final_code = None
MAX_ATTEMPTS = 3

for attempt in range(1, MAX_ATTEMPTS + 1):
    print('#' * 72)
    print(f'# ATTEMPT {attempt}')
    print('#' * 72)

    reply = ask(conversation, system=REACT_SYSTEM)
    print(reply)

    generated = extract_code(reply)
    passed, report = run_acceptance(generated)

    print('\n' + '-' * 72)
    print('OBSERVATION (real harness output)')
    print('-' * 72)
    print(report)
    print()

    transcript.append({'attempt': attempt, 'reply': reply, 'code': generated,
                       'passed': passed, 'report': report})
    final_code = generated

    if passed:
        print(f'All tests green on attempt {attempt}. Stopping the loop.')
        break

    conversation = f"""OBSERVATION from the test harness on your previous ACTION:

{report}

The failing cases above are the real behavior of your code. Diagnose each failure specifically in
THOUGHT, then output the complete corrected module in ACTION. Do not change anything that is
already passing.

The original task, unchanged:
{TASK}"""
else:
    print(f'Hit the {MAX_ATTEMPTS}-attempt ceiling.')

########################################################################
# ATTEMPT 1
########################################################################
THOUGHT:
  We need to implement `calculate_payouts(sales: list[dict]) -> dict` following all specified business rules and error handling.
  Money handling uses `decimal.Decimal` quantized to 2 places (`0.01`).
  Input validation must be strict and raise informative exceptions for invalid input structures, missing keys, wrong types, or out-of-spec values.
  Consignor payout rates: 65% for items with `sale_price >= 200`, 60% otherwise.
  Only payable items (`status == "sold"` and `days_since_sale >= 7`) contribute to output.
  Consignors with 0 payable items must be excluded from the returned dictionary.

PLAN:
  1. Validate input parameter `sales` (must be a list).
  2. For each record in `sales`:
     - Validate keys (`item_id`, `consignor_id`, `sale_price`, `status`, `days_since_sale`) and their types/values.
     - Parse `sale_p

In [26]:
print(final_code)

from decimal import Decimal, ROUND_HALF_UP
from typing import Dict, List, Any

_CENT = Decimal("0.01")
_TWO_HUNDRED = Decimal("200")
_RATE_HIGH = Decimal("0.65")
_RATE_LOW = Decimal("0.60")
_VALID_STATUSES = {"sold", "returned", "pending"}

def _quantize_money(amount: Decimal) -> Decimal:
    return amount.quantize(_CENT, rounding=ROUND_HALF_UP)

def _validate_sale_record(record: dict) -> None:
    if not isinstance(record, dict):
        raise TypeError(f"Expected dict sale record, got {type(record).__name__}")
    
    required_keys = {"item_id", "consignor_id", "sale_price", "status", "days_since_sale"}
    missing = required_keys - set(record.keys())
    if missing:
        raise ValueError(f"Sale record missing required keys: {missing}")
    
    if not isinstance(record["item_id"], str):
        raise TypeError(f"item_id must be str, got {type(record['item_id']).__name__}")
    if not isinstance(record["consignor_id"], str):
        raise TypeError(f"consignor_id must be str, got

In [27]:
# Re-run the suite against the final module, cleanly, so the result is visible on its own.
passed, report = run_acceptance(final_code)
print(report)
print('\nALL TESTS PASSING' if passed else '\nSTILL FAILING')

PASS   empty_input
PASS   basic_split
PASS   return_window
PASS   returned_excluded
PASS   tier_boundary
PASS   rounding_half_up
PASS   negative_price
PASS   two_consignors
PASS   returns_decimal
--- 9/9 passing ---

ALL TESTS PASSING


In [28]:
ns = {}
exec(final_code, ns)
calculate_payouts = ns['calculate_payouts']

SEPTEMBER_SALES = [
    {'item_id': 'VN-4401', 'consignor_id': 'C-207', 'sale_price': '245.00', 'status': 'sold',     'days_since_sale': 21},
    {'item_id': 'VN-4402', 'consignor_id': 'C-207', 'sale_price': '68.00',  'status': 'sold',     'days_since_sale': 14},
    {'item_id': 'VN-4403', 'consignor_id': 'C-207', 'sale_price': '112.50', 'status': 'returned', 'days_since_sale': 19},
    {'item_id': 'VN-4404', 'consignor_id': 'C-118', 'sale_price': '200.10', 'status': 'sold',     'days_since_sale': 9},
    {'item_id': 'VN-4405', 'consignor_id': 'C-118', 'sale_price': '34.00',  'status': 'sold',     'days_since_sale': 2},
    {'item_id': 'VN-4406', 'consignor_id': 'C-330', 'sale_price': '89.99',  'status': 'pending',  'days_since_sale': 0},
    {'item_id': 'VN-4407', 'consignor_id': 'C-412', 'sale_price': '1250.00','status': 'sold',     'days_since_sale': 40},
]

payouts = calculate_payouts(SEPTEMBER_SALES)

print(f"{'CONSIGNOR':<12}{'ITEMS':>7}{'GROSS':>12}{'PAYOUT':>12}")
print('-' * 43)
for cid in sorted(payouts):
    row = payouts[cid]
    print(f"{cid:<12}{row['item_count']:>7}{'$' + str(row['gross_sales']):>12}{'$' + str(row['payout']):>12}")
print('-' * 43)
print(f"{'TOTAL':<12}{sum(r['item_count'] for r in payouts.values()):>7}"
      f"{'$' + str(sum(r['gross_sales'] for r in payouts.values())):>12}"
      f"{'$' + str(sum(r['payout'] for r in payouts.values())):>12}")
print('\nC-330 is absent because that sale is still pending. Correct.')

CONSIGNOR     ITEMS       GROSS      PAYOUT
-------------------------------------------
C-118             1     $200.10     $130.07
C-207             2     $313.00     $200.05
C-412             1    $1250.00     $812.50
-------------------------------------------
TOTAL             4    $1763.10    $1142.62

C-330 is absent because that sale is still pending. Correct.


In [29]:
for t in transcript:
    score = t['report'].strip().splitlines()[-1]
    fails = [l for l in t['report'].splitlines() if l.startswith(('FAIL', 'ERROR'))]
    print(f"Attempt {t['attempt']}: {score}")
    for f in fails:
        print('   ', f)
    print()

Attempt 1: --- 9/9 passing ---

